In [ ]:
import pandas as pd, csv, sys, os
import numpy as np
from pathlib import Path
from datetime import datetime
from unidecode import unidecode

src = Path.cwd().parent/"src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

pd.set_option('display.max_columns', None)

In [ ]:

from routes.Paths import (
    Ruta_arbol_tipificacion,
    Ruta_global, Ruta_local, 
    Ruta_masterskill, 
    Ruta_opl,
    Ruta_busquedad_contacto,
    Ruta_crm,
    Ruta_festivos
)
from utils.functions import detect_delimiter
from utils.Processing_OPL import data_opl
from utils.Processing_MasterSkill import limpiar_masterskill
from utils.Data_BusquedaContacto.a_carga_data import load_contacts
from utils.Data_BusquedaContacto.a_carga_data import load_contacts
from utils.Processing_Arbol import limpiar_df_arbol

In [ ]:
df_contactos = load_contacts(Ruta_busquedad_contacto)
df_opl = data_opl(pd.read_excel(Ruta_opl, sheet_name="Combinado"))
df_master = limpiar_masterskill(pd.read_excel(Ruta_masterskill, sheet_name="TB"))
df_contactos = df_contactos[df_contactos["id de contacto"].notna()]

df_festivos = (
    pd.to_datetime(
        pd.read_csv(Ruta_festivos)["Fecha"],
        format="%d/%m/%Y"
    )
    .drop_duplicates()
    .values.astype("datetime64[D]")
)
df_arbol = limpiar_df_arbol(pd.read_excel(Ruta_arbol_tipificacion, sheet_name='Tipificación Vista360'))
today = pd.Timestamp.today().normalize()

01_01_2026_Digital.csv omitido. Columnas faltantes: {'marca de tiempo de conexion a agente', 'marca de tiempo de desconexion'}
01_01_2026_Voz.csv omitido. Columnas faltantes: {'marca de tiempo de conexion a agente', 'marca de tiempo de desconexion'}
01_02_2026_Digital.csv omitido. Columnas faltantes: {'marca de tiempo de conexion a agente', 'marca de tiempo de desconexion'}
01_02_2026_Voz.csv omitido. Columnas faltantes: {'marca de tiempo de conexion a agente', 'marca de tiempo de desconexion'}
01_03_2026 Digital.csv omitido. Columnas faltantes: {'marca de tiempo de conexion a agente', 'marca de tiempo de desconexion'}
01_03_2026 Voz.csv omitido. Columnas faltantes: {'marca de tiempo de conexion a agente', 'marca de tiempo de desconexion'}
01_04_2026 Digital.csv omitido. Columnas faltantes: {'marca de tiempo de conexion a agente', 'marca de tiempo de desconexion'}
01_04_2026 Voz.csv omitido. Columnas faltantes: {'marca de tiempo de conexion a agente', 'marca de tiempo de desconexion'}


KeyError: 'id de contacto'

In [ ]:
def load_files(folder_path: Path)-> pd.DataFrame:
    require_columns = {
        "RADICADO",
        "PRODUCTO",
        "TIPO",
        "SUBTIPO",
        "ID_LLAMADA_DE_CONTACTO",
        "SE_RESOLVIÓ_EL_CONTACTO",
        "MOTIVO_NO_FCR",
        "NOMBRE_ALIADO",
        "FECHA_DE_INICIO",
        "FECHA_FIN"
    }
    dfs = []
    folder_path = Path(folder_path)

    for file in folder_path.iterdir():
        if not file.is_file():
            continue
        try:
            if file.suffix.lower() == ".csv":
                delimiter = detect_delimiter(file)
                df = pd.read_csv(file, delimiter=delimiter)
            else:
                continue
            columns_validation = require_columns - set(df.columns)
            if columns_validation:
                print(
                    f"Archivo {file} omitido."
                    f"Ausencia de las siguientes columnas {columns_validation}."
                )
                continue
            df["FILE_NAME"] = file.name
            df["CREATION_FILE_DATE"] = datetime.fromtimestamp(file.stat().st_atime)
            dfs.append(df)
        except Exception as e:
            print(f"el archivo {file.name} no pudo ser proceado por el error: {e}.")
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True).drop_duplicates()

df_crm = load_files(Ruta_crm)

In [ ]:
excluir_registros = ["CHAT DE PRUEBA", "LLAMADA DE PRUEBA", "PRUEBAS PCO"]
df_crm = df_crm[~df_crm["SUBTIPO"].isin(excluir_registros)]
columnas_crm = [
    "RADICADO",
    "ESTADO",
    "TIPO_PRI",
    "PRODUCTO",
    "TIPO",
    "SUBTIPO",
    "NÚMERO_CUENTA",
    "TIPO_DE_LLAMADA",
    "ES_CLIENTE_APP",
    "NÚMERO_DOCUMENTO_DEL_CLIENTE",
    "NOMBRE_CLIENTE",
    "NOMBRE_ALIADO",
    "SOLUCIÓN",
    "NOMBRE_DE_AGENTE_ASIGNADO_AL_CASO",
    "FECHA_DE_INICIO",
    "EN_DIAS",
    "SLA_DIAS",
    "FECHA_FIN",
    "VALOR_PUNTOS",
    "VALOR_DINERO",
    "MONTO_MÍNIMO",
    "CANAL_DE_ATENCIÓN",
    "ESTADO_GESTIÓN",
    "NÚMERO_PEDIDO",
    "NOMBRE_ASESOR",
    "DEVOLUCIÓN",
    "SE_RESOLVIÓ_EL_CONTACTO",
    "MOTIVO_NO_FCR",
    "CONTACTO_DE_CLIENTE",
    "ID_LLAMADA_DE_CONTACTO",
    "OFRECIMIENTO_EFECTIVO",
    "SKILL",
    "PROMESA_DE_VENTA",
    "MOTIVO_NO_OFRECIMIENTO",
    "MOTIVO_NO_ACEPTACIÓN",
    "FECHA_ÚLTIMO_CAMBIO",
    "MÉTODO_DE_APROBACIÓN_DE_VALIDACIÓN",
    "CÓDIGO_DE_VALIDACIÓN",
    "FILE_NAME",
    "CREATION_FILE_DATE"
]

df_crm = df_crm[columnas_crm]
for col in columnas_crm:
    df_crm[col] = (
        df_crm[col].astype("string")
        .str.strip()
        .map(lambda x: unidecode(x) if pd.notna(x) else x)
        .str.title()
    )



In [ ]:
condiciones_canal =[
    df_crm["CANAL_DE_ATENCIÓN"].isin(["Whatsapp", "Chat", "Pagina Web"]),
    df_crm["CANAL_DE_ATENCIÓN"].isin(["Telefonico"]),
    df_crm["CANAL_DE_ATENCIÓN"].isin(["Correo Electronico"])
]
resultado_canal = [
    "Chat",
    "Voz",
    "Correo"
]

df_crm["CANAL_DE_ATENCIÓN"] = np.select(condiciones_canal, resultado_canal, default="Otros")


In [ ]:
columnas_fecha = [
    "FECHA_DE_INICIO",
    "FECHA_FIN",
    "FECHA_ÚLTIMO_CAMBIO",
    "CREATION_FILE_DATE"
    ]
for col in columnas_fecha:
    df_crm[col] = pd.to_datetime( df_crm[col], utc=True, errors="coerce").dt.tz_localize(None)

In [ ]:
columnas_numero = [
    "EN_DIAS",
    "SLA_DIAS",
    "VALOR_PUNTOS",
    "VALOR_DINERO"
]

for col in columnas_numero:
    df_crm[col] = ( pd.to_numeric(df_crm[col], errors='coerce').fillna(0).astype("int"))

df_crm["START_DATE"] = df_crm["FECHA_DE_INICIO"].dt.date
df_crm["END_DATE"] = df_crm["FECHA_FIN"].dt.date

In [ ]:
df_crm = df_crm.merge(
    df_arbol[
        [
            "IdTipificacion",
            "Nivel_1",
            "Nivel_2",
            "Nivel_3",
            "SPC/FCR",
            "ANS_Dias",
            "Tipo_Dia",
            "SLA_Dia",
            "SLA_TipoDia"
        ]
    ],
    how="left",
    left_on=["NIVEL_1", "NIVEL_2", "NIVEL_3"],
    right_on=["Nivel_1", "Nivel_2", "Nivel_3"],
    ).drop(columns=["Nivel_1", "Nivel_2", "Nivel_3"])

KeyError: 'NIVEL_1'

In [ ]:
validacion_canal = [ f.title() for f in ["chat", "voz"] ]

validacion_nivel_2 = [ f.title() for f in 
    [
        "Aclaracion",
        "Peticiones",
        "Quejas",
        "Reclamos",
        "Registro",
        "Seguridad"
    ]
]

validacion_nivel_3 = [ f.title() for f in [ "seguimiento y/o solucion de pqr" ]]

validacion_skill = [f.title() for f in 
    [
        "buzon_tienda_online",
        "personas_back office_outbound",
        "empresas_back office_outbound",
        "personas_tentativa_de_fraude",
        "buzon_clientes",
        "monitoreo_voz_outbound",
        "monitoreo_digital_outbound",
        "monitoreo_digital _outbound",
        "negocios_comprobante",
        "buzon_empresas",
        "empresas_tentativa_de_fraude",
        "personas_voz_outbound personas",
        "empresas_voz_outbound empresas",
        "buzon_tienda_aliados",
        "buzon_proteccion_de_datos",
    ]
]

validacion_motivo_no_fcr = [ f.title() for f in 
    [
        "cliente no continua en la interaccion/chat",
        "se contacta tercero",
        "fuera de horario",
    ]
]

condiciones_fcr = (
    df_crm["CANAL_DE_ATENCIÓN"].isin(validacion_canal)
    & df_crm["NIVEL_2"].isin(validacion_nivel_2)
    & ~df_crm["NIVEL_3"].isin(validacion_nivel_3)
    & ~df_crm["TIPO_DE_LLAMADA"].isin(["Contacto Salida"])
    & ~df_crm["SKILL"].isin(validacion_skill)
    & ~df_crm["MOTIVO_NO_FCR"].isin(validacion_motivo_no_fcr)
)

df_crm["CONTADOR_FCR"] = np.where(condiciones_fcr, 1, 0)

In [ ]:
condicion_mal_tipificado = (
    (df_crm["SPC/FCR"].notna()) &
    (df_crm["CONTADOR_FCR"] > 0) &
    (df_crm["SE_RESOLVIÓ_EL_CONTACTO"] != df_crm["SPC/FCR"])
)

df_crm["MAL_TIPIFICADO_FCR"] = np.where(condicion_mal_tipificado, 1, 0)

In [ ]:
validacion_habiles = df_crm["Tipo_Dia"].eq("Habiles")

start = (
    df_crm.loc[validacion_habiles, "FECHA_DE_INICIO"]
    .dt.normalize()
    .values
    .astype("datetime64[D]")
)

end = (
    df_crm.loc[validacion_habiles, "FECHA_FIN"]
    .fillna(today)
    .dt.normalize()
    .values
    .astype("datetime64[D]")
)

dias_habiles = np.busday_count(
    start + np.timedelta64(1, "D"),
    end + np.timedelta64(1, "D"),
    holidays = df_festivos
)

df_crm["DIAS_CIERRE"] = df_crm["EN_DIAS"]
df_crm.loc[validacion_habiles, "DIAS_CIERRE"] = dias_habiles
df_crm["DIAS_CIERRE"] =(
    pd.to_numeric(df_crm["DIAS_CIERRE"],errors="coerce")
    .fillna(0)
    .clip(lower=0)
    )

In [ ]:
condicion = (
    df_crm["DIAS_CIERRE"]
    .le(df_crm["ANS_Dias"])
    .fillna(False)
)

df_crm["CUMPLIMIENTO_ANS"] = np.where(
    df_crm["FECHA_FIN"].notna(),
    np.where(
        condicion,
        "Cumple",
        "No Cumple"
    ),
    pd.NA
)

In [ ]:
# Casos abiertos (FECHA_FIN vacía)
mask_sin_fin = df_crm["FECHA_FIN"].isna()

inicio_abiertos = (
    df_crm.loc[mask_sin_fin, "FECHA_DE_INICIO"]
    .dt.normalize()
    .to_numpy()
    .astype("datetime64[D]")
)

fin_today = np.datetime64(pd.Timestamp(today).normalize(), "D")

# Días hábiles
dias_habiles_gestion = np.busday_count(
    inicio_abiertos + np.timedelta64(1, "D"),
    fin_today + np.timedelta64(1, "D"),
    holidays=df_festivos
)

# Días corridos
dias_corridos_gestion = (fin_today - inicio_abiertos).astype(int)

mask_habiles_abiertos = (
    df_crm.loc[mask_sin_fin, "Tipo_Dia"]
    .eq("Habiles")
    .to_numpy()
)

resultado_final = np.where(
    mask_habiles_abiertos,
    dias_habiles_gestion,
    dias_corridos_gestion
)
resultado_final = np.nan_to_num(resultado_final, nan=0)

dias_ns = (
    pd.to_numeric(
        df_crm.loc[mask_sin_fin, "ANS_Dias"],
        errors="coerce"
    )
    .fillna(0)
    .to_numpy()
)

porc_gestion = np.divide(
    resultado_final,
    dias_ns,
    out=np.full(resultado_final.shape, np.nan, dtype=float),
    where=dias_ns != 0
)
condiciones_estado = [
    (porc_gestion > 0) & (porc_gestion <= 0.5),
    (porc_gestion > 0.5) & (porc_gestion <= 0.75),
    (porc_gestion > 0.75) & (porc_gestion <= 1),
    (porc_gestion > 1)
]

resultados_estado = [
    "Verde",
    "Amarillo",
    "Rojo",
    "Negro"
]

estado_abiertos = np.select(
    condiciones_estado,
    resultados_estado,
    default=pd.NA
)

# Inicializa toda la columna
df_crm["ESTADO_GESTION"] = pd.NA

# Asigna únicamente a registros abiertos
df_crm.loc[mask_sin_fin, "ESTADO_GESTION"] = estado_abiertos

In [ ]:
inicio_all = df_crm["FECHA_DE_INICIO"].dt.normalize().values.astype("datetime64[D]")
fin_today = np.datetime64(pd.Timestamp(today).normalize(), "D")
dias_transcurridos = (fin_today - inicio_all).astype(int)

meta = pd.to_numeric( df_crm["ANS_Dias"], errors="coerce" )

dias_restantes = (meta - dias_transcurridos).astype("float64")
condiciones_grupo = [
    dias_restantes < 0,
    dias_restantes <= 1,
    dias_restantes <= 3,
    dias_restantes <= 5,
    dias_restantes <= 7,
    dias_restantes <= 9,
]
resultados_grupo = [
    "Vencido",
    "0 - 1 Día",
    "2 - 3 Días",
    "4 - 5 Días",
    "6 - 7 Días",
    "8 - 9 Días",
]

grupo_dias = np.select(condiciones_grupo, resultados_grupo, default=">=10 Días")

# Solo aplica a casos sin FECHA_FIN; el resto queda en blanco
df_crm["GRUPO_DIAS_CIERRE"] = np.where(mask_sin_fin, grupo_dias, pd.NA)

In [ ]:
df_crm["ID_LLAMADA_DE_CONTACTO"] = (df_crm["ID_LLAMADA_DE_CONTACTO"].str.lower())
df_crm = df_crm.merge(
    df_contactos[["id de contacto", "agente", "duración del contacto"]],
    how="left",
    left_on="ID_LLAMADA_DE_CONTACTO",
    right_on="id de contacto"
).drop(columns=["id de contacto"])

df_crm = df_crm.merge(
    df_opl[["Nombre_Agente", "Correo_contratista"]],
    how="left",
    left_on="NOMBRE_ASESOR",
    right_on="Nombre_Agente"
).drop(columns=["Nombre_Agente"])

df_crm["agente"] = (df_crm["agente"].combine_first(df_crm["Correo_contratista"]))
df_crm = df_crm.drop(columns=["Correo_contratista"])

df_crm = df_crm.merge(
    df_opl[["Id_opl" ,"Correo_contratista"]],
    how="left",
    left_on="agente",
    right_on="Correo_contratista"
).drop(columns=["Correo_contratista"])

In [ ]:
df_master["Colas"] = (df_master["Colas"].str.title())
df_crm = df_crm.merge(
    df_master[
        [
            "IdMasterSkill", "Canal", "Colas" 
    ]
],
how="left",
left_on=["CANAL_DE_ATENCIÓN", "SKILL"],
right_on=["Canal", "Colas"]
).drop(columns=[ "Canal", "Colas"])

In [ ]:
df_crm = df_crm.rename(columns={
    "ANS_Dias":"ANS_DIAS",
    "Tipo_Dia":"TIPO_DIA",
    "agente":"CORREO_AGENTE",
    "duración del contacto":"DURACION_CONTACTO"
})
df_crm["HORA_INICIO"] = df_crm["FECHA_DE_INICIO"].dt.time
df_crm.to_csv(os.path.join(Ruta_global, "04.Data_CRM.csv"), index=False, encoding="utf-8")
df_crm.to_csv(os.path.join(Ruta_local, "04.Data_CRM.csv"), index=False, encoding="utf-8")

In [ ]:
df_crm.to_parquet(os.path.join(Ruta_global, "04.Data_CRM.parquet"), index=False)